### Agent Evaluation

For RAG, we used the A->Q->A' setup:

* A = original answer in the FAQ
* Q = generated question from this answer
* A' = answer produced by our RAG system

For agents, we use the same setup. A' comes from an agent instead of a fixed RAG pipeline.

We also save the trajectory. Here, the trajectory means only the tool calls the agent made before producing the final answer.

#### Loading the data

Use the same ground truth questions:

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("./ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

Load the FAQ documents and the search index:

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

Create a lookup table:

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["doc_id"]] = doc

#### Running the agent

Reuse the ToyAIKit agent from module 01. It handles the agent loop and stores the full message history.

First, set up the model clients:


In [6]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

Define the search tool:

In [7]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

Create the runner:

In [8]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

The result contains:

* last_message: the final response
* all_messages: the full message history
* cost: the cost of all LLM calls in this run

Run it for one ground truth question:

In [9]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

Look at the full message history:

In [10]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Is it okay to start the course late if I just found it now?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"start the course late found it now late start enrollment begin anytime"}', call_id='call_HHUqCi7EYroKUEWcDVVIHA6N', name='search', type='function_call', id='fc_03511295200f7948006a5636562b1c81a1a7ee6be1acaf6936', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_HHUqCi7EYroKUEWcDVVIHA6N',
  'output': '[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [

For this lesson, the trajectory is only the tool calls. We don't need to send the full message history to the judge.

**Extract the function name and arguments**

In [11]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

For this example:

In [12]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"start the course late found it now late start enrollment begin anytime"}'}]

You should see something like this:

In [13]:
[
    {
        "name": "search",
        "arguments": "{\"query\":\"own pace certificate at the end self-paced course certificate\"}"
    }
]

[{'name': 'search',
  'arguments': '{"query":"own pace certificate at the end self-paced course certificate"}'}]

Get the original answer:

In [14]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

Save the A->Q->A' record and the trajectory:

In [15]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'Is it okay to start the course late if I just found it now?',
 'answer_agent': 'Yes — you can start the course whenever you want. The videos and GitHub materials are available, so if you just found it now, you can begin from the start at your own pace.\n\nA couple of notes:\n- Homework submissions are only possible while the form is open.\n- There are no late submissions once a homework form closes.\n\nIf you want, I can also tell you the best way to catch up quickly.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"start the course late found it now late start enrollment begin anytime"}'}],
 'cost': Decimal('0.00148725'),
 'document': '74eb249bbf'}

The answer_agent field is what we evaluate with the LLM judge. The tool_calls field lets the judge see how the agent got there.

### Processing multiple questions

Create a function that processes one ground truth record:

In [16]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

Run it for a small sample in parallel:

In [17]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

Turn it into a dataframe:

In [18]:
df_agent = pd.DataFrame(agent_answers)

Calculate the total cost:

In [19]:
df_agent["cost"].sum()

Decimal('0.06521250')

Save the results:

In [20]:
df_agent.to_csv("./agent-answers.csv", index=False)

Now we have the same A->Q->A' data as before, plus the tool calls for each agent run.

We generated this file for the course materials on May 29, 2026. The run used 50 ground truth questions. ToyAIKit tracks the agent cost for each run, so we can sum the cost column directly.

The total agent cost was $0.06993300, about 7 cents.

If you don't want to run the agent yourself, download the file we prepared:

PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main


wget -O data/agent-answers.csv ${PREFIX}/04-evaluation/data/agent-answers.csv

**Then load it:** 
(This could be a new notebook)

In [14]:
#if you want to start from here, and not load all the cells before
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [8]:
import pandas as pd

df_agent = pd.read_csv("./agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

Our judge can look at both:

* whether answer_agent matches answer_orig
* whether the tool calls look reasonable for the question

This lets us evaluate the final answer and the agent behavior in one place.

#### Judging answers and trajectories

A good trajectory is not just "many tool calls". A good trajectory uses the available tools in a way that helps answer the question.

For our search agent, a good trajectory has these properties:

* The search query is relevant to the user question
* The query includes the important keywords from the question
* The agent avoids duplicate searches with the same arguments
* If it searches more than once, the next query is a useful refinement
* It usually uses 1 search call
* 2-3 calls can be okay for harder questions
* More than 3 search calls needs a clear reason
* The tool calls support the final answer
* The agent does not stop too early or keep searching without a reason

Now define a judge output type with two scores:


In [9]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

The judge instructions:

In [10]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

Define the judge function:

In [11]:
import ast
import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        # After a CSV round-trip, tool_calls is a Python repr string
        # (single quotes), not JSON. Try JSON first, fall back to
        # ast.literal_eval for the Python-literal case.
        try:
            tool_calls = json.loads(tool_calls)
        except json.JSONDecodeError:
            tool_calls = ast.literal_eval(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

Test it on one agent result:

In [12]:
#testing: checking content of agent_answers[0]
agent_answers[0]

{'question': 'Is it okay to start the course late if I just found it now?',
 'answer_agent': 'Yes — you can start the course whenever you want. The videos and GitHub materials are available, so you can begin late and work through the lessons at your own pace.\n\nA couple of notes:\n- Homework can only be submitted while the submission form is open.\n- There are no late submissions or individual deadline extensions once the form is closed.\n\nIf you want, I can also point you to the best way to catch up quickly.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': '[{\'name\': \'search\', \'arguments\': \'{"query":"start the course late found it now late enrollment start late okay"}\'}]',
 'cost': 0.00150525,
 'document': '74eb249bbf'}

In [15]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent answer matches the key point of the ground truth: starting late is okay. It also correctly adds that to receive a certificate, work must be submitted while submissions are still being accepted, which is consistent with the original answer. The extra details about videos, GitHub materials, and no late submissions are not in the ground truth, but they do not conflict with it.', answer_score='good', trajectory_reasoning='The search query was relevant to the user’s question about starting the course late and included key ideas like starting late and whether it is okay. Only one search call was made, which is reasonable. The call supported the final answer.', trajectory_score='good')

When the answer is bad, the trajectory score tells us whether the problem started with tool use. 

If the answer is bad but the trajectory is good, the model may have used the retrieved context poorly. 

If both are bad, the agent likely searched for the wrong thing. It may also have stopped too early.

#### Running the agent judge
Run the judge for all agent answers:



In [16]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

Use the same parallel helper:

In [18]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

Split the results:

In [19]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

Create a dataframe:

In [20]:
df_agent_eval = pd.DataFrame(agent_evaluations)

Calculate the judge cost from the token usage:

In [21]:
calc_total_price(usages)

0.054799500000000015

Check the answer scores:

In [22]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    46
bad      4
Name: count, dtype: int64

Check the trajectory scores:

In [23]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    50
Name: count, dtype: int64

Save the judge results:

In [25]:
df_agent_eval.to_csv("./agent-evaluations.csv", index=False)

**We generated this file for the course materials on May 29, 2026. The run judged 50 agent answers.**

The answer scores were:

Good: 45
Bad: 5

The trajectory scores were:

Good: 49
Bad: 1

The judge token usage was:

Input tokens: 29,228
Output tokens: 6,984
Cost with the prices above: $0.053349, about 5 cents

If you don't want to run the judge yourself, download the file we prepared:

PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main


wget -O data/agent-evaluations.csv ${PREFIX}/04-evaluation/data/agent-evaluations.csv


Evaluation frameworks

For production systems, consider using evaluation frameworks that make it easier to manage test datasets, run evaluations, and track results:

* Ragas: a framework for evaluating RAG systems with metrics like faithfulness, answer relevance, and context precision
* DeepEval: provides built-in metrics for RAG evaluation including hallucination detection and answer relevance
* TruLens: instruments your LLM app and tracks quality metrics

These frameworks implement many of the concepts we covered here and add visualizations and experiment tracking.

Monitoring

Online evaluation (monitoring) is what you do after deploying your system.

Key approaches:

* User feedback: thumbs up/down buttons to collect signal
* Logging: record queries, retrieved documents, and answers
* Dashboards: track metrics over time to spot degradation
* Alerts: get notified when metrics drop below a threshold
* Monitoring is covered in more detail in module 05.